# Customer Segmentation using K-Means

In this lab, we will use K-Means clustering to group customers with similar characteristics.

The main idea is to understand how K-Means forms groups and how those groups can be used for a real-world problem such as customer segmentation.

## 1. Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 2. Load the dataset

Upload `customer_segmentation.csv` to Colab before running this cell.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
df = pd.read_csv('customer_segmentation.csv')
df.head()

## 3. Understand the data

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

## 4. Check for missing values

K-Means works with numerical values, so it is useful to check the data before building the model.

In [ ]:
df.isnull().sum()

## 5. Select the features

For this lab, we will use Annual Income and Spending Score to understand customer behavior.

These two features make the clusters easy to visualize.

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]
X.head()

## 6. Visualize the data before clustering

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X['Annual Income (k$)'], X['Spending Score (1-100)'])
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customers Before Clustering')
plt.show()

## 7. Scale the features

The two features are on different ranges. Scaling puts them on a comparable scale before calculating distances.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled[:5]

## 8. Find a suitable value of K

K-Means needs us to decide how many clusters we want.

We can use the Elbow Method to compare different values of K.

In [ ]:
inertia = []

for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia.append(model.inertia_)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), inertia, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(range(2, 11))
plt.show()

## 9. Build the K-Means model

From the elbow plot, choose a reasonable K. For this dataset, we will start with K = 5.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

clusters[:10]

## 10. Add the cluster labels to the dataset

In [ ]:
df['Cluster'] = clusters
df.head()

## 11. Visualize the customer segments

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    X['Annual Income (k$)'],
    X['Spending Score (1-100)'],
    c=clusters,
    cmap='viridis',
    s=60
)

centers = scaler.inverse_transform(kmeans.cluster_centers_)

plt.scatter(
    centers[:, 0],
    centers[:, 1],
    c='red',
    marker='X',
    s=200,
    label='Centroids'
)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customer Segments using K-Means')
plt.legend()
plt.show()

## 12. Look at the cluster centers

The centroid represents the center of each customer group.

In [ ]:
centers_df = pd.DataFrame(
    centers,
    columns=['Annual Income (k$)', 'Spending Score (1-100)']
)

centers_df

## 13. Check how many customers are in each cluster

In [ ]:
df['Cluster'].value_counts().sort_index()

## 14. Interpret the clusters

Now we connect the clusters with customer behavior.

- Low income + low spending → customers with lower spending activity
- Low income + high spending → customers who spend more despite lower income
- High income + low spending → customers with potential for targeted offers
- High income + high spending → valuable customers

The exact interpretation depends on the cluster centers in our dataset.

## 15. Evaluate the clustering

The silhouette score gives us an idea of how well the data points fit their assigned clusters.

In [ ]:
score = silhouette_score(X_scaled, clusters)
print("Silhouette Score:", round(score, 3))

## 16. Try another K

Changing K changes the number of customer groups. Let's compare a few values using the silhouette score.

In [ ]:
for k in range(2, 8):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f"K = {k}, Silhouette Score = {score:.3f}")

## 17. Final takeaway

K-Means does not know the customer categories beforehand.

It finds groups by looking at the distance between data points and their centroids.

In this example, we used customer income and spending behavior to create meaningful customer segments that could be useful for marketing and business decisions.

### Practice Task

Try changing K and observe how the customer groups change.

Then think about this question:

**Which customer segment would you target for a premium marketing campaign, and why?**